## **Feature Importance Analysis via Permutation Importance**

### Note: Notebook 2 and 3 must must be run first on the same kernel.

---

### **Overview**

This notebook analyzes the directional impact of features on the remote work prediction using sklearn's built-in permutation_importance applied to the best Random Forest model from the tuning notebook. It will use the model that had a majority of unneeded features removed to speed up runtime.

**permutation_importance** works by shuffling one feature column at a time across 10  repeats and measuring how much the model's accuracy drops on average. A large drop means the feature is important. 

A Mann-Whitney U test will then be performed on each to be able to find if it leads to remote or non-remote conditions.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import mannwhitneyu
from sklearn.inspection import permutation_importance
from scipy.stats import ttest_1samp
from sklearn.metrics import accuracy_score


In [2]:
%store -r
print("Variables restored successfully.")
print(f"RF model: {rf}")
print(f"X_test shape: {X_test_scaled.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"Surviving features: {len(current_features)}")


Variables restored successfully.
RF model: RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=42)
X_test shape: (9839, 395)
y_test shape: (9839,)
Surviving features: 198


In [3]:

feature_names = current_features

# Best number of features: 198

X_array = X_clean.columns.tolist()
feature_indexes = [X_array.index(f) for f in feature_names]
X_final = pd.DataFrame(X_test_scaled[:, feature_indexes], columns=feature_names)

n_features = len(feature_names)
test_loops = 10
total_runs = n_features * test_loops

print(f"Features: {n_features}")
print(f"Repeats: {test_loops}")
print(f"Total Runs: {total_runs}")

# Progress-tracking
call_counter = [0]

def tracking_scorer(estimator, X, y):
    call_counter[0] += 1
    current = call_counter[0]
    feat_num = (current - 1) // test_loops + 1
    repeat_num = (current - 1) % test_loops + 1
    remaining = total_runs - current
    if repeat_num == 1:
        print(f"  [{current:>4}/{total_runs}] Feature {feat_num:>3}/{n_features} — {remaining} evaluations left")
    return accuracy_score(y, estimator.predict(X))

result = permutation_importance(rf, X_final, y_test, n_repeats=test_loops,
                                random_state=42, n_jobs=1, scoring=tracking_scorer)

result_df = pd.DataFrame({
    'Feature': feature_names,
    'Mean_Drop': result.importances_mean,
    'Std_Drop': result.importances_std
})

result_df = result_df.sort_values('Mean_Drop', ascending=False).reset_index(drop=True)

print(f"\nDone. {call_counter[0]} total evaluations completed.")
print(f"\nTop 5 features by permutation importance:")
print(f"{'Rank':<6} {'Feature':<45} {'Mean Drop':>10} {'Std':>8}")
print("-" * 75)
for i, row in result_df.head(5).iterrows():
    print(f"{i+1:<6} {row['Feature']:<45} {row['Mean_Drop']:>10.4f} {row['Std_Drop']:>8.4f}")


Features: 198
Repeats: 10
Total Runs: 1980
  [   1/1980] Feature   1/198 — 1979 evaluations left
  [  11/1980] Feature   2/198 — 1969 evaluations left
  [  11/1980] Feature   2/198 — 1969 evaluations left
  [  21/1980] Feature   3/198 — 1959 evaluations left
  [  21/1980] Feature   3/198 — 1959 evaluations left
  [  31/1980] Feature   4/198 — 1949 evaluations left
  [  31/1980] Feature   4/198 — 1949 evaluations left
  [  41/1980] Feature   5/198 — 1939 evaluations left
  [  41/1980] Feature   5/198 — 1939 evaluations left
  [  51/1980] Feature   6/198 — 1929 evaluations left
  [  51/1980] Feature   6/198 — 1929 evaluations left
  [  61/1980] Feature   7/198 — 1919 evaluations left
  [  61/1980] Feature   7/198 — 1919 evaluations left
  [  71/1980] Feature   8/198 — 1909 evaluations left
  [  71/1980] Feature   8/198 — 1909 evaluations left
  [  81/1980] Feature   9/198 — 1899 evaluations left
  [  81/1980] Feature   9/198 — 1899 evaluations left
  [  91/1980] Feature  10/198 — 1889 ev

## **Top 20 Features by Permutation Importance**


In [4]:
top20 = result_df.head(20).sort_values('Mean_Drop')

fig = go.Figure()
fig.add_trace(go.Bar(
    x=top20['Mean_Drop'],
    y=top20['Feature'],
    orientation='h',
    error_x=dict(type='data', array=top20['Std_Drop'].tolist(), visible=True),
    marker_color='royalblue'
))
fig.update_layout(
    title='Random Forest — Top 20 Features by Permutation Importance (Mean Accuracy Drop)',
    xaxis_title='Mean Accuracy Drop',
    yaxis_title='Feature',
    template='plotly_white',
    height=600
)
fig.show()

print(f"{'Rank':<6} {'Feature':<45} {'Mean Drop':>10} {'Std':>8}")
print("-" * 75)
for i, row in result_df.head(20).iterrows():
    print(f"{i+1:<6} {row['Feature']:<45} {row['Mean_Drop']:>10.4f} {row['Std_Drop']:>8.4f}")


Rank   Feature                                        Mean Drop      Std
---------------------------------------------------------------------------
1      org_size                                          0.0690   0.0023
2      work_exp                                          0.0096   0.0019
3      region_northern_america                           0.0089   0.0013
4      employment_employed                               0.0086   0.0015
5      region_eastern_europe                             0.0066   0.0010
6      employment_independent                            0.0058   0.0007
7      years_code                                        0.0043   0.0011
8      collab_jira                                       0.0037   0.0010
9      region_south_america                              0.0030   0.0006
10     platform_amazon_web_services_aws                  0.0024   0.0011
11     profession_professional dev                       0.0024   0.0008
12     lang_python                              

## **Statistical Significance of Features**

For each of the all features, a **one-sample t-test** is run on the 10 repeat importance scores against a null hypothesis of zero to see if it will consistently be different. A feature is considered statistically significant if p < 0.05.


In [5]:

all_names = result_df['Feature'].tolist()
sig_results = []

for feature in all_names:
    feature_index = feature_names.index(feature)
    # Get their scores
    repeat_scores = result.importances[feature_index]

    t_stat, p_val = ttest_1samp(repeat_scores, popmean=0)
    mean_drop = repeat_scores.mean()
    std_drop = repeat_scores.std()

    sig_results.append({
        'Feature': feature,
        'Mean_Drop': mean_drop,
        'Std_Drop': std_drop,
        't_stat': t_stat,
        'p_value': p_val,
        'Significant': 'Yes' if p_val < 0.05 else 'No'
    })

significant_df = pd.DataFrame(sig_results).sort_values('Mean_Drop', ascending=False)

print(f"{'Rank':<6} {'Feature':<45} {'Mean Drop':>10} {'Std':>8} {'t-stat':>8} {'p-value':>10} {'Sig':>7}")
print("-" * 100)
for rank, (_, row) in enumerate(significant_df.iterrows(), 1):
    print(f"{rank:<6} {row['Feature']:<45} {row['Mean_Drop']:>10.4f} {row['Std_Drop']:>8.4f} {row['t_stat']:>8.3f} {row['p_value']:>10.4f} {row['Significant']:>7}")

total_significant = (significant_df['Significant'] == 'Yes').sum()
print(f"\n{'='*100}")
print(f"Total features tested   : {len(significant_df)}")
print(f"Statistically significant (p < 0.05) : {total_significant} / {len(significant_df)}")


Rank   Feature                                        Mean Drop      Std   t-stat    p-value     Sig
----------------------------------------------------------------------------------------------------
1      org_size                                          0.0690   0.0023   90.507     0.0000     Yes
2      work_exp                                          0.0096   0.0019   14.839     0.0000     Yes
3      region_northern_america                           0.0089   0.0013   20.819     0.0000     Yes
4      employment_employed                               0.0086   0.0015   16.906     0.0000     Yes
5      region_eastern_europe                             0.0066   0.0010   20.704     0.0000     Yes
6      employment_independent                            0.0058   0.0007   26.211     0.0000     Yes
7      years_code                                        0.0043   0.0011   11.426     0.0000     Yes
8      collab_jira                                       0.0037   0.0010   11.091     0.000

## **Direction Analysis**

To analyze the features, they were compared against the media of the test set where if it is higher, then it would be a **positive** predictor while lower would be a **negative predictor**

A **Mann-Whitney U test** was used to check whether the difference is statistically significant (p < 0.05).


In [8]:

all_features = result_df['Feature'].tolist()
y_test_array = np.array(y_test)

direction_results = []

# Get the feature value and compare it against the median of the test set
for feature in all_features:
    feature_index = feature_names.index(feature)
    feature_val = X_final.iloc[:, feature_index].to_numpy().astype(np.float64)
    median_val = np.median(feature_val)

    high_marker = feature_val > median_val
    low_marker  = feature_val <= median_val

    remote_rate_high = y_test_array[high_marker].mean() if high_marker.sum() > 0 else np.nan
    remote_rate_low  = y_test_array[low_marker].mean()  if low_marker.sum()  > 0 else np.nan

    if high_marker.sum() > 0 and low_marker.sum() > 0:
        i, p_val = mannwhitneyu(feature_val[y_test_array == 1], feature_val[y_test_array == 0], alternative='two-sided')
    else:
        p_val = np.nan

    direction = 'Positive' if remote_rate_high > remote_rate_low else 'Negative'
    mean_drop = result_df.loc[result_df['Feature'] == feature, 'Mean_Drop'].values[0]

    direction_results.append({
        'Feature': feature,
        'Mean_Drop': mean_drop,
        'Remote_Rate_High': remote_rate_high,
        'Remote_Rate_Low': remote_rate_low,
        'Direction': direction,
        'p_value': p_val,
        'Significant': 'Yes' if (not np.isnan(p_val) and p_val < 0.05) else 'No'
    })

dir_df = pd.DataFrame(direction_results)

print(f"{'Rank':<6} {'Feature':<45} {'Direction':<28} {'p-value':>10} {'Sig':>5}")
print("-" * 100)
for rank, (i, row) in enumerate(dir_df.iterrows(), 1):
    print(f"{rank:<6} {row['Feature']:<45} {row['Direction']:<28} {row['p_value']:>10.4f} {row['Significant']:>5}")

total_significant = (dir_df['Significant'] == 'Yes').sum()
positive_total = ((dir_df['Significant'] == 'Yes') & (dir_df['Direction'].str.contains('Positive'))).sum()
negative_total = ((dir_df['Significant'] == 'Yes') & (dir_df['Direction'].str.contains('Negative'))).sum()

print(f"\n{'='*100}")
print(f"Total features: {len(dir_df)}")
print(f"Statistically significant: {total_significant} / {len(dir_df)}")
print(f"Positive: {positive_total}")
print(f"Negative: {negative_total}")


Rank   Feature                                       Direction                       p-value   Sig
----------------------------------------------------------------------------------------------------
1      org_size                                      Positive                         0.0000   Yes
2      work_exp                                      Positive                         0.0000   Yes
3      region_northern_america                       Positive                         0.0000   Yes
4      employment_employed                           Negative                            nan    No
5      region_eastern_europe                         Positive                         0.0000   Yes
6      employment_independent                        Negative                         0.0000   Yes
7      years_code                                    Positive                         0.0000   Yes
8      collab_jira                                   Positive                         0.0000   Yes
9      r

In [ ]:

# Merge direction info into top 20
top20_dir = result_df.head(20).merge(
    dir_df[['Feature', 'Direction', 'Significant']],
    on='Feature'
)

## Signed importance: negative for Negative predictors
top20_dir['Signed_Importance'] = top20_dir.apply(
    lambda r: r['Mean_Drop'] if r['Direction'] == 'Positive' else -r['Mean_Drop'], axis=1
)

# Sort ascending so highest positive is at top, most negative is at bottom
top20_dir = top20_dir.sort_values('Signed_Importance', ascending=True)

colors = ['tomato' if d == 'Negative' else 'royalblue' for d in top20_dir['Direction']]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=top20_dir['Signed_Importance'],
    y=top20_dir['Feature'],
    orientation='h',
    marker_color=colors
))

fig.add_vline(x=0, line_dash='solid', line_color='black', line_width=1)

fig.update_layout(
    title=dict(
        text='Top 20 Features — Signed Permutation Importance<br><sup>Blue = Positive (→ Remote) | Red = Negative (→ Non-Remote)</sup>',
        x=0,
        xanchor='left'
    ),
    xaxis=dict(
        title='Signed Mean Accuracy Drop',
        tickangle=0
    ),
    yaxis_title='Feature',
    template='plotly_white',
    height=650,
    margin=dict(b=60)
)
fig.show()

print(f"{'Rank':<6} {'Feature':<45} {'Mean Drop':>10} {'Direction':<12} {'Sig':>5}")
print("-" * 85)
for rank, (_, row) in enumerate(top20_dir.sort_values('Signed_Importance', ascending=False).iterrows(), 1):
    print(f"{rank:<6} {row['Feature']:<45} {row['Mean_Drop']:>10.4f} {row['Direction']:<12} {row['Significant']:>5}")


Rank   Feature                                        Mean Drop Direction      Sig
-------------------------------------------------------------------------------------
1      org_size                                          0.0690 Positive       Yes
2      work_exp                                          0.0096 Positive       Yes
3      region_northern_america                           0.0089 Positive       Yes
4      region_eastern_europe                             0.0066 Positive       Yes
5      years_code                                        0.0043 Positive       Yes
6      collab_jira                                       0.0037 Positive       Yes
7      region_south_america                              0.0030 Positive       Yes
8      platform_amazon_web_services_aws                  0.0024 Positive       Yes
9      lang_python                                       0.0020 Positive       Yes
10     devtype_backend developer                         0.0020 Positive       Yes
1